![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Manual Indicator Warm-Up Research

Warm manually managed daily indicators from history before building indicator and signal series.

### Set Up QuantBook

Create a daily SPY subscription for the manual indicator updates.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.DAILY)

### Warm Manual Indicators

Create an EMA and RSI, store them in one list, and warm both from the same history request.

In [ ]:
ema = ExponentialMovingAverage(10)
rsi = RelativeStrengthIndex(12)
indicators = [ema, rsi]
# Warm the manually managed indicators with daily history.
history = qb.history[TradeBar](equity, max(indicator.warm_up_period for indicator in indicators), Resolution.DAILY)
for bar in history:
    for indicator in indicators:
        indicator.update(bar)

### Verify Readiness

Confirm each manual indicator is ready before building the series.

In [ ]:
print(f"EMA samples: {ema.samples}")
print(f"EMA ready: {ema.is_ready}")
print(f"EMA value: {ema.current.value:.2f}")
print(f"RSI samples: {rsi.samples}")
print(f"RSI ready: {rsi.is_ready}")
print(f"RSI value: {rsi.current.value:.2f}")
assert all([indicator.samples >= indicator.warm_up_period for indicator in indicators])
assert all([indicator.is_ready for indicator in indicators])

### Build Time Series

Replay a longer history window and store the raw EMA and RSI values.

In [ ]:
series_ema = ExponentialMovingAverage(10)
series_rsi = RelativeStrengthIndex(12)
series_indicators = [series_ema, series_rsi]
value_rows = []
signal_rows = []
for bar in qb.history[TradeBar](equity, 120, Resolution.DAILY):
    for indicator in series_indicators:
        indicator.update(bar)
    if not all([indicator.is_ready for indicator in series_indicators]):
        continue
    value_rows.append({"time": bar.end_time, "ema": series_ema.current.value, "rsi": series_rsi.current.value})
    signal_rows.append({
        "time": bar.end_time,
        "signal": 1 if all([indicator.current.value > indicator.previous.value for indicator in series_indicators]) else 0
    })

indicator_values = pd.DataFrame(value_rows).set_index("time")
indicator_values.tail()

### Signal Series

Display the 1/0 signal generated when both indicators are increasing.

In [ ]:
signals = pd.DataFrame(signal_rows).set_index("time")
signals.tail()